# Choose the customers to score

Supply four customer IDs to a custom prediction table, including a customer whose graph record already has a label.
This Kumo Engine module uses synthetic retail data in a Databricks Python notebook.

See [supplied prediction rows](https://docs.nvidia.com/sdgm/kumo-engine/user-code-guide/train-a-model/predictive-queries#use-supplied-prediction-rows) for the input contract.

## Setup

Use Databricks Serverless GPU environment 6 or Databricks Runtime 17.3 LTS ML on GPU compute.
Install the latest Kumo Engine package, then restart Python so the notebook uses the installed package.

In [ ]:
%pip install "kumo-engine[databricks]"

In [ ]:
dbutils.library.restartPython()

<br>

## Set the example parameters

Choose the local working directory and the four customer IDs to score. Each run creates its own directory; the demonstration model uses one FAST experiment.


In [ ]:
from pathlib import Path
import tempfile
import pandas as pd
from IPython.display import display
import kumo.engine as kumo
from kumoapi.train import TrainingTableSpec
from kumoapi.typing import Stype

WORK_ROOT = Path("/tmp/kumoai-local")
SCORE_IDS = [5, 40, 42, 49]

WORK_ROOT.mkdir(parents=True, exist_ok=True)
run_dir = Path(tempfile.mkdtemp(prefix="custom-prediction-", dir=WORK_ROOT))
data_dir = run_dir / "source"
data_dir.mkdir()
kumo.configure(scratch_dir=run_dir / "scratch")
print("Kumo Engine:", kumo.__version__)
print("Example directory:", run_dir)

<br>

## Generate synthetic example data

Run the next cell to create 50 customers, 150 transactions, and 30 supplied training labels with fixed train, validation, and test splits.
You can skim the generation code; the Kumo workflow starts below.
Customer IDs 0 to 39 have a churn label, while IDs 40 to 49 do not.


In [ ]:
customers_frame = pd.DataFrame({
    "ID": range(50),
    "CHURNED": pd.array([bool(i % 2) for i in range(40)] + [None] * 10, dtype="boolean"),
    "VISITS": [float(i % 7) for i in range(50)],
})
transactions_frame = pd.DataFrame({
    "TX_ID": range(150),
    "CUSTOMER_ID": [i % 50 for i in range(150)],
    "AMOUNT": [float(5 + (i * 7) % 40) for i in range(150)],
    "TIMESTAMP": pd.Timestamp("2025-01-01") + pd.to_timedelta([i % 30 for i in range(150)], unit="D"),
})
customers_frame.to_parquet(data_dir / "customers.parquet", index=False)
transactions_frame.to_parquet(data_dir / "transactions.parquet", index=False)
assert transactions_frame["CUSTOMER_ID"].isin(customers_frame["ID"]).all()
display(customers_frame.head())
print("Customers:", len(customers_frame), "Transactions:", len(transactions_frame))

training_rows = pd.DataFrame({
    "ENTITY1": range(30),
    "TARGET1": [bool(i % 2) for i in range(30)],
    "SPLIT": [0] * 20 + [1] * 6 + [2] * 4,
})
training_rows.to_parquet(data_dir / "training.parquet", index=False)

<br>

## Build the graph

Connect each transaction's customer key to `customers.ID` and validate the graph.
The customer table has no time column, so this static prediction task needs only customer IDs.


In [ ]:
connector = kumo.FileConnector(root_dir=str(data_dir))
customers = kumo.Table.from_source_table(
    connector["customers.parquet"], primary_key="ID",
).infer_metadata()
transactions = kumo.Table.from_source_table(
    connector["transactions.parquet"], primary_key="TX_ID", time_column="TIMESTAMP",
).infer_metadata().with_column("CUSTOMER_ID", stype=Stype.ID)
graph = kumo.Graph(
    tables={"customers": customers, "transactions": transactions},
    edges=[kumo.Edge(src_table="transactions", fkey="CUSTOMER_ID", dst_table="customers")],
)
graph.validate()

Graph visualization is a preview unavailable in this public release; leave this optional cell commented until it is available in the installed package.
See [graph definitions](https://docs.nvidia.com/sdgm/kumo-engine/user-code-guide/set-up-a-graph/graphs#inspect-a-graph-definition) for the tables and relationships.

In [ ]:
# Preview: graph visualization is unavailable in the supported package.
# graph.visualize(show_cols=True, show_samples=True)


<br>

## Prepare a demonstration model

Create a snapshot and validate the binary churn query.
This model supplies scores for the population example; it does not measure predictive quality.


In [ ]:
snapshot = graph.snapshot()
query = kumo.PredictiveQuery(graph, "PREDICT customers.CHURNED FOR EACH customers.ID")
query.validate()
print("Task:", query.get_task_type())

Fit the supplied labels using 20 training rows, 6 validation rows, and 4 test rows.
Use one FAST experiment without refitting so the module stays focused on selecting scoring rows.

In [ ]:
from kumoapi.model_plan import RunMode

spec = TrainingTableSpec(split_col="SPLIT")
training_table = kumo.TrainingTable.from_source_table(
    connector["training.parquet"], spec, pq=query, snapshot=snapshot,
)
model_plan = query.suggest_model_plan(snapshot, train_table_spec=spec, run_mode=RunMode.FAST)
model_plan.training_job.num_experiments = 1
model_plan.training_job.refit_trainval = False
model_plan.training_job.refit_full = False
model_plan.training_job.disable_explain = True
trainer = kumo.Trainer(model_plan=model_plan)
fit_result = trainer.fit(training_table, snapshot=snapshot)
print("Synthetic demonstration fit completed.")

<br>

## Batch predictions

Customer 5 already has a label; customers 40, 42, and 49 do not.
A custom prediction table includes the supplied customers regardless of missing labels.
Write their primary keys as `ENTITY1`, omit target columns, and reuse the fitted trainer's generation plan.


In [ ]:
scoring_rows = pd.DataFrame({"ENTITY1": SCORE_IDS})
scoring_rows.to_parquet(data_dir / "scoring.parquet", index=False)
prediction_table = kumo.PredictionTable.from_source_table(
    connector["scoring.parquet"],
    pq=query,
    snapshot=snapshot,
    plan=trainer.training_table_generation_plan,
)
assert prediction_table.row_count == len(scoring_rows)
print("Requested rows:", len(scoring_rows))
print("Materialized population rows:", prediction_table.row_count)

Score the four supplied customers with the fitted model.
The 0.5 threshold controls the returned binary label.

In [ ]:
result = trainer.predict(
    prediction_table,
    snapshot=snapshot,
    binary_classification_threshold=0.5,
)

Check the returned IDs and save the scores to a local Parquet file.
This binary task returns one row per customer; `True_PROB` is the positive-label probability and `TARGET_PRED` is true when it exceeds 0.5.
Scores are a synthetic demonstration only; customer 5 was also in the training set.

In [ ]:
scored = result.frame.sort_values("ENTITY").reset_index(drop=True)
assert scored["ENTITY"].tolist() == sorted(SCORE_IDS)
assert scored["ENTITY"].is_unique
print("Output rows:", result.num_rows)
display(scored[["ENTITY", "TARGET_PRED", "True_PROB"]])
predictions_path = run_dir / "predictions.parquet"
scored.to_parquet(predictions_path, index=False)
print("Saved predictions:", predictions_path)

<br>

## Reject target columns

A prediction population contains inputs, not outcomes.
The next cell deliberately supplies `TARGET1` and checks that validation asks you to remove it.


In [ ]:
invalid_rows = scoring_rows.assign(TARGET1=False)
invalid_rows.to_parquet(data_dir / "invalid-scoring.parquet", index=False)
try:
    kumo.PredictionTable.from_source_table(
        connector["invalid-scoring.parquet"], pq=query, snapshot=snapshot,
        plan=trainer.training_table_generation_plan,
    )
except ValueError as error:
    assert "target" in str(error).lower()
    print(error)
else:
    raise AssertionError("Prediction rows with a target column should be rejected.")

<br>

## Apply this to another task

Use actual snapshot primary keys in `ENTITY1`.
Temporal tasks and static tasks on timestamped entity tables also require a timezone-naive UTC `TIMESTAMP` for each supplied row.

For multiclass tasks, each customer can produce several output rows, one per retained class with `CLASS` and `SCORE` columns.
`num_classes_to_return=2` requests up to two classes per population row; binary classification does not accept this option.
See [task-specific prediction outputs](https://docs.nvidia.com/sdgm/kumo-engine/user-code-guide/advanced-prediction-configurations#task-specific-output-options) before assuming that population and output counts match.

Continue with [custom training tables](https://docs.nvidia.com/sdgm/kumo-engine/user-code-guide/train-a-model/predictive-queries#use-custom-training-tables) to supply your own labels and splits.
Keep `run_dir` while inspecting this example; its files remain on local storage.
